# Compute the coverage for a gene
February 29, 2024

Load the replicate 1 and replicate 2 datasets, for each chromosome, for each gene
compute the MNase-seq coverage of the gene.

- Inputs: MNase-seq data for each gene for all timepoints for both replicates
- Outputs: A csv file of the coverage per each replicate for each gene.


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [18]:
geneset = pd.read_csv('data/reference_data/geneset_nondub_w_prom_genebodies.csv').set_index('orf_name')


In [10]:
from cc_src.chromatin_model import ChromatinModel
from src.config import load_yl_rg1_vst_config

Loading MNase reads for YAL067C/SEO1...None 1
Loading chromosome reads: 1
Done.


In [42]:
from src.timer import Timer

def get_coverage_for_replicate(replicate):
    
    timer = Timer()
    all_coverages = geneset[[]].copy()
    all_coverages['coverage'] = np.nan

    config = load_yl_rg1_vst_config(replicate)
    chromatin_model = ChromatinModel(config)
    n = len(all_coverages)        
    i = 0
    for chrom in range(1, 17):
        chrom_geneset = geneset[geneset.chr == chrom]

        for orf_name, row in chrom_geneset.iterrows():
            chromatin_model.load_mnase_gene(orf_name, log=False)
            chromatin_model.create_deconvolution_bins()
            deconv_bins = chromatin_model.deconv_hist_unflattened
            coverage = (deconv_bins.sum(axis=0).sum(axis=0) > 0).mean()
            all_coverages.loc[orf_name, 'coverage'] = coverage

            if i % 100 == 0:
                print(f"{i}/{n} - {timer.get_time()}")
            i += 1

    return all_coverages


In [ ]:
coverage_1 = get_coverage_for_replicate(1)


0/5774 - 00:00:01.35
100/5774 - 00:00:36.90
200/5774 - 00:01:17.12
300/5774 - 00:01:57.52
400/5774 - 00:02:38.13
500/5774 - 00:03:18.99
600/5774 - 00:03:50.86
700/5774 - 00:04:43.54
800/5774 - 00:05:37.33
900/5774 - 00:06:31.19
1000/5774 - 00:07:24.63
1100/5774 - 00:08:17.80
1200/5774 - 00:09:10.37
1300/5774 - 00:10:02.44
1400/5774 - 00:10:56.38
1500/5774 - 00:11:33.08
1600/5774 - 00:12:09.76
1700/5774 - 00:12:45.45
1800/5774 - 00:13:24.71
1900/5774 - 00:14:09.99
2000/5774 - 00:14:55.23
2100/5774 - 00:15:41.07
2200/5774 - 00:16:27.01
2300/5774 - 00:17:12.68
2400/5774 - 00:17:52.15
2500/5774 - 00:18:28.22
2600/5774 - 00:19:06.07
2700/5774 - 00:19:40.12
2800/5774 - 00:20:19.84
2900/5774 - 00:20:59.39
3000/5774 - 00:21:38.88
3100/5774 - 00:22:18.32
3200/5774 - 00:23:00.94
3300/5774 - 00:23:40.61


In [ ]:
coverage_2 = get_coverage_for_replicate(2)


In [ ]:
coverage_1.head()